---
---
---

# Regression Models

---
---
---

<br>

> **_Abstract:_** This notebook focuses on the training DataFrame, which is the primary dataset used for building and evaluating machine learning models. The training DataFrame contains various features and a target variable that we aim to predict. We will perform feature engineering, preprocessing, and target analysis to prepare the data for modeling. Finally, we will establish baseline models to evaluate our initial performance.

> **_Abstract:_** ...

> **_Table of Contents:_**
>
> 1. DataFrame
> 2. Training
>     - Train-Test Split
>     - Dummy Regressor (Baseline)
>     - Preprocessor
>     - Dummy Regressor (Baseline)
>     - Hist Gradient Boosting Regressor
>     - Random Forest Regressor
> 3. Testing
>     - Dummy Regressor (Baseline)
>     - Hist Gradient Boosting Regressor
>     - Random Forest Regressor
> 7. Conclusion

<br>

> | **Language: python@3.11** |
> | - |

<br>

> | **_Source:_** [**_EnergyPlus AI: Regression Models_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-regression-models) |
> | - |

<br>

> | **_Libraries:_** [**_github.com/robertovicario/EnergyPlus-AI/notebook/lib_**](https://github.com/robertovicario/EnergyPlus-AI/tree/main/notebook/lib) |
> | - |

<br>

---
---
---

## Dependencies

### Packages

In [1]:
from loguru import logger
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import os
import pandas as pd
import warnings

from lib import ml_metrics as metrics_lib

# -------------------------

warnings.filterwarnings('ignore')

### Paths

In [2]:
DATA_PATH = os.path.join('..', 'data')

### Configurations

In [3]:
config = {
    'model': {
        'name': 'EnergyPlusAI_Model',
        'task': 'Regression',
        'dataset': 'udse.parquet',
        'target': 'EUI'
    },
    'training': {
        'df_rows': 0,
		'df_columns': 0,
        'frac': 1.0,
        'test_size': 0.2,
        'random_state': 42,
        'shuffle': True
    }
}

## DataFrame

> **_Source:_** [**_EnergyPlus AI: Training DataFrame_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-training-dataframe)

In [4]:
df = pd.read_parquet(
    os.path.join(DATA_PATH, 'udse.parquet'),
    engine='fastparquet'
)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255036 entries, 0 to 260747
Data columns (total 25 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   Building-Type    255036 non-null  category
 1   Climate-Zone     255036 non-null  category
 2   Total-Area       255036 non-null  float64 
 3   Floor-Area       255036 non-null  float64 
 4   Num-Floors       255036 non-null  float64 
 5   Building-Depth   255036 non-null  float64 
 6   Building-Length  255036 non-null  float64 
 7   Building-Height  255036 non-null  float64 
 8   Floor-Height     255036 non-null  float64 
 9   WWR              255036 non-null  float64 
 10  Wall-R-Value     255036 non-null  category
 11  Roof-R-Value     255036 non-null  category
 12  Window-U-Value   255036 non-null  category
 13  SHGC             255036 non-null  category
 14  HVAC             255036 non-null  object  
 15  EUI              255036 non-null  float64 
 16  HVAC-Type        255036 n

## Training

### Train-Test Split

In [5]:
# Feature Selection
X = df.drop(columns=['EUI', 'EUI-Dist'])
y = df['EUI']

# Fractional Sampling
frac = config['training']['frac']
subset = int(len(X) * frac)
X = X.iloc[:subset]
y = y.iloc[:subset]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['training']['test_size'],
    random_state=config['training']['random_state'],
    shuffle=config['training']['shuffle']
)

# -------------------------

logger.debug(f"DF{'':<4}(Shape): {X.shape}")
logger.debug(f"TRAIN{'':<1}(Shape): {X_train.shape}")
logger.debug(f"TEST{'':<2}(Shape): {X_test.shape}")

2026-04-24 11:02:27.045 | DEBUG    | __main__:<module>:21 - DF    (Shape): (255036, 23)
2026-04-24 11:02:27.046 | DEBUG    | __main__:<module>:22 - TRAIN (Shape): (204028, 23)
2026-04-24 11:02:27.046 | DEBUG    | __main__:<module>:23 - TEST  (Shape): (51008, 23)


### Preprocessor

In [6]:
bool_cols = X_train.select_dtypes(include=['bool']).columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['bool', 'object', 'category']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('bool', 'passthrough', bool_cols),
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore'
        ), cat_cols)
    ]
)
display(preprocessor)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bool', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. 

### Dummy Regressor (Baseline)

In [7]:
from sklearn.dummy import DummyRegressor
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
display(dummy)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None


### Hist Gradient Boosting Regressor

In [8]:
hgb = HistGradientBoostingRegressor(
    categorical_features='from_dtype',
    random_state=config['training']['random_state']
)
hgb.fit(X_train, y_train)
display(hgb)

ValueError: could not convert string to float: 'PVAV-Gas_Boiler-_Reheat'

### Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(
            n_jobs=config['training']['n_jobs'],
            random_state=config['training']['random_state']
        ))
    ]
)
rf.fit(X_train, y_train)
display(rf)

## Testing

### Dummy Regressor (Baseline)

In [ ]:
# Prediction
y_pred_train = dummy.predict(X_train)
y_pred_test = dummy.predict(X_test)

# Metric
metrics_dummy = {}
metrics_lib.compute_metrics_reg(metrics_dummy, y_train, y_pred_train, set='TRAIN')
metrics_lib.compute_metrics_reg(metrics_dummy, y_test, y_pred_test, set='TEST')
metrics_lib.out_metrics(metrics_dummy, ml_task='regression')

In [ ]:
from sklearn.neural_network import MLPRegressor
nn = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', MLPRegressor(
            random_state=config['training']['random_state']
        ))
    ]
)
nn.fit(X_train, y_train)
display(nn)

# Prediction
y_pred_train = nn.predict(X_train)
y_pred_test = nn.predict(X_test)

# Metric
metrics_nn = {}
metrics_lib.compute_metrics_reg(metrics_nn, y_train, y_pred_train, set='TRAIN')
metrics_lib.compute_metrics_reg(metrics_nn, y_test, y_pred_test, set='TEST')
display(df['EUI'].describe())
metrics_lib.out_metrics(metrics_nn, ml_task='regression')

### Hist Gradient Boosting Regressor

In [ ]:
# Prediction
y_pred_train = hgb.predict(X_train)
y_pred_test = hgb.predict(X_test)

# Metric
metrics_hgb = {}
metrics_lib.compute_metrics_reg(metrics_hgb, y_train, y_pred_train, set='TRAIN')
metrics_lib.compute_metrics_reg(metrics_hgb, y_test, y_pred_test, set='TEST')
display(df['EUI'].describe())
metrics_lib.out_metrics(metrics_hgb, ml_task='regression')

### Random Forest Regressor

In [ ]:
# Prediction
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)

# Metric
metrics_rf = {}
metrics_lib.compute_metrics_reg(metrics_rf, y_train, y_pred_train, set='TRAIN')
metrics_lib.compute_metrics_reg(metrics_rf, y_test, y_pred_test, set='TEST')
display(df['EUI'].describe())
metrics_lib.out_metrics(metrics_rf, ml_task='regression')

## Conclusion

> The training dataset contains a **_solid mix of categorical and numerical features_**, along with a sufficient number of observations to support the development of robust machine learning models.

> The presence of outliers in the target variable indicates that energy consumption behavior is not uniform across buildings. In particular, low- and high-consumption buildings appear to **_follow distinct patterns_**, suggesting that a single model may struggle to capture these differences effectively.

> To explore this, baseline models were implemented as a reference point for evaluating potential improvements. The results indicate that a **_multi-stage_** approach combining a **_classification model_** to distinguish between low and high energy consumption buildings, followed by dedicated **_regression models_** to predict `EUI` for each group shows strong promise.

> Building on these findings, the **_next steps_** involve developing more advanced models, such as **_tree-based methods_** and **_neural networks_**, along with systematic hyperparameter tuning to further enhance performance beyond the baseline.

> Further experimentation and model development can be found in the following notebooks:
>
> - [**_EnergyPlus AI: Energy Model_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-energy-model)